AF 7-2025

# Young people and time alone

[description]

In [ ]:
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

source = "..//data//source//"
processed = "..//data//processed//"

long_glob = glob.glob(processed + "0324//atus*.csv")
print(long_glob)

['..//data//processed//0324/atussum-0324.csv', '..//data//processed//0324/atusresp-0324.csv', '..//data//processed//0324/atusact-0324.csv', '..//data//processed//0324/atuscps-0324.csv', '..//data//processed//0324/atusrost-0324.csv', '..//data//processed//0324/atuswho-0324.csv']


/home/afan/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


: 

In [ ]:
a = {}

for path in long_glob: 

    name = path.split("/")[-1][4:-9]

    a[name] = pd.read_csv(path)


/tmp/ipykernel_17150/1160267113.py:7: DtypeWarning: Columns (25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  a[name] = pd.read_csv(path)


In [ ]:
a['act'].head()

In [ ]:
# groupby caseid, activity, and just add the amount of time doing that activity 

act_time = a['act'].groupby(["TUCASEID", "TRCODEP", "TRTIER1P", "TRTIER2P"])['TUACTDUR24'].agg(['count', 'sum']).reset_index()
act_time = act_time.rename(columns = {"count": "n_activity", "sum": "minutes"})

# making sure it's adding up properly 
act_time.groupby("TUCASEID")['minutes'].sum().value_counts()


In [ ]:
print(act_time.shape)
act_time.head()

In [ ]:
# pick the right respondents -- we only care about respondents under

sum_under_35 = a['sum'][a['sum']['TEAGE']<35]
sum_under_35.head()

In [ ]:
sum_under_35.TEAGE.value_counts()

In [ ]:
sum_under_35.TUYEAR.value_counts().sort_index()

In [ ]:
sum_under_35.head()

In [ ]:
sum_variables = ["TUCASEID", # ID
                 "TUYEAR", # Year
    "TESCHENR", # School enrollment
    "TRSPPRES", # Presence of spouse
    "TUDIARYDAY", # day of week
    "TESEX", # Sex
    "TRERNWA", # Weekly earnings 
    "TEHRUSLT", # Usual weekly hours of work 
    "TESCHLVL", # school level
    "TUFNWGTP", # Weight
    "TEAGE", # Age 
    "PTDTRACE", # Race
    "PEHSPNON", # Hispanic Origin
    "PEEDUCA", # Educational attainment
    "GTMETSTA", # new metro status
    "GEMETSTA" # old metro status
]

sum_to_merge = sum_under_35[sum_variables]
sum_to_merge.head()

In [ ]:
# each row is a case-activity
m = pd.merge(sum_to_merge, act_time, on = "TUCASEID",  how = "left", validate="1:m")
print(m.shape)
m.head()

# Doing the analyses by demographic group

* What's taking up a lot less time now? 

In [ ]:
# sum the weights for each year 
weights_sum = sum_to_merge.groupby("TUYEAR")["TUFNWGTP"].sum().reset_index()
weights_sum.columns = ['year', 'weights_sum']
weights_sum.head()

In [ ]:
def weighted_average_pivot(df, df_resp, weight_column, year_col, other_weighted_group_cols, group_cols, agg_col): 

    # get the total of the weights by group 
    weighted_group_cols = year_col + other_weighted_group_cols
    weights_by_group = df_resp.groupby(weighted_group_cols)[weight_column].agg(['sum', 'count']).reset_index()
    # get the weighted sum for each group
    groupby_cols = weighted_group_cols + group_cols
    avg_by_code = df.groupby(groupby_cols)[[agg_col, weight_column]].apply(lambda df: (df[agg_col] @ df[weight_column]) * 7/60).reset_index()

    # merge and divide out to get the averages
    abc_w = pd.merge(avg_by_code, weights_by_group, on = weighted_group_cols, how = 'left', validate='m:1')
    abc_w['avg_min_weighted'] = abc_w[0]/abc_w['sum']

    act_avg_pivot = abc_w.pivot_table(index = group_cols + other_weighted_group_cols, columns=year_col, values='avg_min_weighted', aggfunc='mean').reset_index()
    act_avg_pivot['change'] = act_avg_pivot[2024]/act_avg_pivot[2004]

    return act_avg_pivot


In [ ]:
activity_code_vars = ['TRCODEP', 'TRTIER1P', 'TRTIER2P']

with pd.ExcelWriter(processed + 'avg_hours_per_week.xlsx') as writer:

    for var in activity_code_vars: 

        f = weighted_average_pivot(m, sum_to_merge, 'TUFNWGTP', ['TUYEAR'], [], [var], 'minutes')

        f.to_excel(writer, sheet_name = var, index =None)

sex 
educational attainment
metro status 

In [ ]:
with pd.ExcelWriter(processed + 'avg_hours_per_week_by_sex.xlsx') as writer:

    for var in activity_code_vars: 

        f = weighted_average_pivot(m, sum_to_merge, 'TUFNWGTP', ['TUYEAR'],  ['TESEX'], [var], 'minutes')

        f.to_excel(writer, sheet_name = var, index=None)

In [ ]:
with pd.ExcelWriter(processed + 'avg_hours_per_week_by_sex_race.xlsx') as writer:

    for var in activity_code_vars: 

        f = weighted_average_pivot(m, sum_to_merge, 'TUFNWGTP', ['TUYEAR'],  ['TESEX', 'PTDTRACE'], [var], 'minutes')

        f.to_excel(writer, sheet_name = var, index=None)

In [ ]:
with pd.ExcelWriter(processed + 'avg_hours_per_week_by_sex_ethnicity.xlsx') as writer:

    for var in activity_code_vars: 

        f = weighted_average_pivot(m, sum_to_merge, 'TUFNWGTP', ['TUYEAR'],  ['TESEX', 'PEHSPNON'], [var], 'minutes')

        f.to_excel(writer, sheet_name = var, index=None)

## Are people spending more time alone? 

Doing what? 

In [ ]:
a['who'].head()

In [ ]:
alone = a['who'][((a['who'].TUWHO_CODE==18)| (a['who'].TUWHO_CODE==19))]

alone.shape

In [ ]:
# each row is an activity done alone by a person

alone_merged = pd.merge(alone, a['act'], on = ['TUCASEID', 'TUACTIVITY_N'], how = 'left', validate = "1:1").groupby(["TUCASEID", "TRCODEP", "TRTIER1P", "TRTIER2P"])['TUACTDUR24'].agg(['count', 'sum']).reset_index()
alone_merged = alone_merged.rename(columns = {"count": "n_activity", "sum": "minutes"})
print(alone_merged.shape)
alone_merged.head()

In [ ]:
# merge with the 25 and under crowd so that... we get the weights 

alone_25 = pd.merge(alone_merged, sum_to_merge, on = "TUCASEID", how = 'right', validate = "m:1")
print(alone_25.shape)
alone_25.head()

In [ ]:
with pd.ExcelWriter(processed + 'alone_hours_per_week.xlsx') as writer:

    for var in activity_code_vars: 

        f = weighted_average_pivot(alone_25, sum_to_merge, 'TUFNWGTP', ['TUYEAR'], [], [var], 'minutes')

        f.to_excel(writer, sheet_name = var, index =None)

In [ ]:
with pd.ExcelWriter(processed + 'alone_hours_per_week_by_sex.xlsx') as writer:

    for var in activity_code_vars: 

        f = weighted_average_pivot(alone_25, sum_to_merge, 'TUFNWGTP', ['TUYEAR'], ['TESEX'], [var], 'minutes')

        f.to_excel(writer, sheet_name = var, index =None)

In [ ]:
weighted_average_pivot(alone_25, sum_to_merge, 'TUFNWGTP', ['TUYEAR'], ['TESEX'], [var], 'minutes')

In [ ]:
test = alone_25.groupby(["TUCASEID", "TUYEAR", "TUFNWGTP"])['minutes'].sum().reset_index()

In [ ]:
sums = test.groupby('TUYEAR').apply(lambda x: np.average(x.minutes, weights=x.TUFNWGTP))
sums.plot()
plt.title("Average minutes spent alone every day, 25 and under")

In [ ]:
sums


In [ ]:
test = alone_25.groupby(["TUCASEID", "TUYEAR", "TUFNWGTP", "TESEX"])['minutes'].sum().reset_index()
sums = test.groupby(['TUYEAR', 'TESEX']).apply(lambda x: np.average(x.minutes, weights=x.TUFNWGTP)).reset_index()
sums.groupby('TESEX')[0].plot()
plt.title("Average time spent alone by sex (blue = M, orange = F)")

In [ ]:
# This is the reason we don't need the fancy weighting function
print(sum_to_merge.TUCASEID.nunique())
print(test.TUCASEID.nunique())